In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder
from sklearn.pipeline import Pipeline 
from scikeras.wrappers import KerasClassifier
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping


I0000 00:00:1788682959.955414    3185 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788682961.404842    3185 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788682964.784322    3185 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


___
# Load The Dataset 
___

In [2]:
data = pd.read_csv("Churn_Modelling.csv")
data

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


___

# PreProcessing Data
___

In [3]:
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis = 1)

In [4]:
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

# use OHE for Geography Feature

from sklearn.preprocessing import OneHotEncoder

ohe_encoder_geo = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

geo_encoded = ohe_encoder_geo.fit_transform(data[["Geography"]])

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=ohe_encoder_geo.get_feature_names_out()
)

data = pd.concat(
    [data, geo_encoded_df],
    axis=1
)


data = data.drop("Geography", axis=1)



In [5]:
data


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


____
# X and Y SPlit 
____

In [6]:
X = data.drop("Exited", axis=1)
y = data["Exited"]

___
# Train Test Split
___

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.20, random_state=42)

___
# Scaler 
___

In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

____
____
# Hyper Parameter Tuning 
___
___

## Define A Function to create a Model & try differnt Paramters

> Use KeraClassifier

In [9]:
def crete_model(neurons= 32, layers = 1):

    model = Sequential
    model.add(
        Dense(neurons, activation="relu", input_shape=(X_train_scaled.shape[1],))          # 1st Layer
    )

    #for Hidden Layers

    for _ in range(layers - 1):
        model.add(
            Dense(neurons, activation="relu")
        )


    #output Layer
    model.add(
        Dense(1,activation="sigmoid")
    )

    # Compile the Model 

    model.compile(
        optimizer = "adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model



In [10]:
# Create a Keras Classifier

model = KerasClassifier(
    layers = 1,
    neurons = 32,
    build_fn=crete_model,
    epochs=50,
    batch_size=10,
    verbose=0
)

In [ ]:
# Define Grid search paramters

param_grid = {
    "neurons" : [16, 32, 64, 128],
    "layers" : [1,2],
    "epochs" : [50, 100]
}


grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv= 3,
    n_jobs=-1
)

grid_result = grid.fit(X_train, y_train)

print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))


I0000 00:00:1788682970.989271    3280 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788682971.083269    3288 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788682971.141283    3285 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788682971.194863    3280 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788682971.203275    3288 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788682971.211440    3279 

____
____
____

# Explanation 
___

```python 

# ============================================================
# HYPERPARAMETER TUNING OF AN ANN USING GRID SEARCH
# ============================================================

# ------------------------------------------------------------
# STEP 1: CREATE A FUNCTION THAT BUILDS OUR ANN MODEL
# ------------------------------------------------------------

# Syntax:
# def function_name(parameter1=default_value, parameter2=default_value):
#
# Here:
#   neurons -> number of neurons in each Dense hidden layer
#   layers  -> number of hidden layers in the ANN
#
# Default values:
#   neurons = 32
#   layers = 1
#
# GridSearchCV will later change these values automatically
# and test different combinations.

def create_model(neurons=32, layers=1):

    # --------------------------------------------------------
    # Create an empty Sequential neural network
    # --------------------------------------------------------
    #
    # Sequential() means:
    # The layers of the neural network will be added one
    # after another in a sequential order.
    #
    # Example:
    #
    # Input -> Dense -> Dense -> Output
    #
    # Syntax:
    # model = Sequential()

    model = Sequential()


    # --------------------------------------------------------
    # STEP 2: ADD THE FIRST HIDDEN LAYER
    # --------------------------------------------------------

    # Syntax:
    #
    # Dense(
    #     number_of_neurons,
    #     activation='activation_function',
    #     input_shape=(number_of_features,)
    # )
    #
    # neurons:
    # Number of neurons in this layer.
    #
    # activation='relu':
    # ReLU activation function is used in the hidden layer.
    #
    # input_shape:
    # Tells the neural network how many input features it will
    # receive.
    #
    # X_train.shape[1]:
    #   shape[0] -> number of rows/samples
    #   shape[1] -> number of columns/features
    #
    # Example:
    #
    # If X_train has:
    #
    #       1000 rows
    #       10 columns
    #
    # Then:
    #
    # X_train.shape
    #       (1000, 10)
    #
    # X_train.shape[1]
    #       10
    #
    # Therefore:
    #
    # input_shape=(10,)

    model.add(
        Dense(
            neurons,
            activation='relu',
            input_shape=(X_train.shape[1],)
        )
    )


    # --------------------------------------------------------
    # STEP 3: ADD ADDITIONAL HIDDEN LAYERS
    # --------------------------------------------------------

    # We already created the FIRST hidden layer above.
    #
    # Therefore, if the user wants:
    #
    # layers = 1
    #
    # We don't need to add another hidden layer.
    #
    # If:
    #
    # layers = 2
    #
    # We need to add 1 additional hidden layer.
    #
    # If:
    #
    # layers = 3
    #
    # We need to add 2 additional hidden layers.
    #
    # That's why we use:
    #
    # range(layers - 1)
    #
    #
    # Example:
    #
    # layers = 1
    # range(0)
    # -> loop runs 0 times
    #
    # layers = 2
    # range(1)
    # -> loop runs 1 time
    #
    # layers = 3
    # range(2)
    # -> loop runs 2 times

    for _ in range(layers - 1):

        # Add another Dense hidden layer.
        #
        # neurons:
        # Number of neurons in this layer.
        #
        # activation='relu':
        # ReLU activation function.

        model.add(
            Dense(
                neurons,
                activation='relu'
            )
        )


    # --------------------------------------------------------
    # STEP 4: ADD THE OUTPUT LAYER
    # --------------------------------------------------------

    # This is a BINARY CLASSIFICATION problem.
    #
    # Therefore, we use:
    #
    # Dense(1)
    #
    # because we need only ONE output neuron.
    #
    # activation='sigmoid'
    #
    # Sigmoid converts the output into a probability between
    # 0 and 1.
    #
    # Example:
    #
    # output = 0.82
    #
    # means approximately:
    #
    # 82% probability of class 1.
    #
    # Usually:
    #
    # probability >= 0.5 -> class 1
    # probability < 0.5  -> class 0
    #
    # Syntax:
    #
    # Dense(1, activation='sigmoid')

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )


    # --------------------------------------------------------
    # STEP 5: COMPILE THE MODEL
    # --------------------------------------------------------

    # Before training the neural network, we must compile it.
    #
    # Syntax:
    #
    # model.compile(
    #     optimizer='...',
    #     loss='...',
    #     metrics=[...]
    # )
    #
    #
    # optimizer='adam'
    # ----------------
    # Adam is the optimization algorithm used to update the
    # weights of the neural network.
    #
    # It controls how the model learns and adjusts its weights.
    #
    #
    # loss='binary_crossentropy'
    # --------------------------
    # Because this is a binary classification problem,
    # binary cross-entropy is used as the loss function.
    #
    # Example:
    #
    # Class 0 -> No disease
    # Class 1 -> Disease
    #
    #
    # metrics=['accuracy']
    # --------------------
    # Accuracy tells us how many predictions were correct.
    #
    # Example:
    #
    # 90 correct predictions out of 100
    #
    # Accuracy = 90%

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )


    # --------------------------------------------------------
    # STEP 6: RETURN THE CREATED MODEL
    # --------------------------------------------------------

    # The function returns the ANN model.
    #
    # Whenever GridSearchCV calls:
    #
    # create_model(...)
    #
    # it receives this compiled model.

    return model


# ============================================================
# STEP 7: CREATE A KERAS CLASSIFIER
# ============================================================

# GridSearchCV from scikit-learn expects an estimator that
# follows the scikit-learn API.
#
# KerasClassifier acts as a bridge between:
#
#       Keras/TensorFlow
#             +
#       Scikit-learn
#
# It allows us to use Keras neural networks with tools such as:
#
#   GridSearchCV
#   RandomizedSearchCV
#   cross_val_score
#
#
# Syntax:
#
# KerasClassifier(
#     build_fn=create_model,
#     epochs=50,
#     batch_size=10,
#     verbose=0
# )
#
#
# build_fn=create_model
# ---------------------
# Tells KerasClassifier which function should be used to
# create the ANN.
#
#
# epochs=50
# ----------
# The model will train for 50 epochs by default.
#
# One epoch means:
# One complete pass through the entire training dataset.
#
#
# batch_size=10
# -------------
# The model processes 10 training samples at a time before
# updating the weights.
#
#
# verbose=0
# ---------
# Controls training output.
#
# verbose=0 -> show nothing
# verbose=1 -> show progress bar
# verbose=2 -> show one line per epoch

model = KerasClassifier(
    build_fn=create_model,
    epochs=50,
    batch_size=10,
    verbose=0
)


# ============================================================
# STEP 8: DEFINE THE HYPERPARAMETER GRID
# ============================================================

# GridSearchCV needs to know which hyperparameters it should
# try.
#
# We create a Python dictionary.
#
# Syntax:
#
# param_grid = {
#     'parameter_name': [value1, value2, value3]
# }
#
#
# IMPORTANT:
#
# The names used here MUST match parameters accepted by the
# KerasClassifier / model-building function.

param_grid = {

    # --------------------------------------------------------
    # NUMBER OF NEURONS
    # --------------------------------------------------------
    #
    # GridSearchCV will test:
    #
    # neurons = 16
    # neurons = 32
    # neurons = 64
    # neurons = 128

    'neurons': [16, 32, 64, 128],


    # --------------------------------------------------------
    # NUMBER OF HIDDEN LAYERS
    # --------------------------------------------------------
    #
    # GridSearchCV will test:
    #
    # layers = 1
    # layers = 2
    #
    # Remember:
    #
    # layers = 1
    # -> one hidden Dense layer
    #
    # layers = 2
    # -> two hidden Dense layers

    'layers': [1, 2],


    # --------------------------------------------------------
    # NUMBER OF EPOCHS
    # --------------------------------------------------------
    #
    # GridSearchCV will test:
    #
    # epochs = 50
    # epochs = 100

    'epochs': [50, 100]
}


# ============================================================
# STEP 9: CREATE GRIDSEARCHCV
# ============================================================

# GridSearchCV automatically tries different combinations
# of hyperparameters.
#
# Syntax:
#
# GridSearchCV(
#     estimator=model,
#     param_grid=param_grid,
#     n_jobs=-1,
#     cv=3
# )
#
#
# estimator=model
# ----------------
# This is the model we want to tune.
#
# Here:
#
# model = KerasClassifier(...)
#
#
# param_grid=param_grid
# ---------------------
# This contains all possible hyperparameter values.
#
#
# n_jobs=-1
# ---------
# Tells scikit-learn to use all available CPU cores.
#
# This can make GridSearchCV faster.
#
#
# cv=3
# ----
# Means 3-fold cross-validation.
#
# The training data is divided into 3 parts.
#
# Example:
#
# Fold 1 -> validation
# Fold 2 + Fold 3 -> training
#
# Then:
#
# Fold 2 -> validation
# Fold 1 + Fold 3 -> training
#
# Then:
#
# Fold 3 -> validation
# Fold 1 + Fold 2 -> training
#
# The scores from the 3 folds are averaged.


grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    n_jobs=-1,
    cv=3
)


# ============================================================
# STEP 10: TRAIN / PERFORM GRID SEARCH
# ============================================================

# This is where the actual hyperparameter tuning happens.
#
# Syntax:
#
# grid.fit(X_train, y_train)
#
#
# X_train
# --------
# Training input/features.
#
#
# y_train
# --------
# Training target/labels.
#
#
# GridSearchCV will:
#
# 1. Take one combination of hyperparameters.
# 2. Build an ANN using those parameters.
# 3. Train the ANN.
# 4. Perform 3-fold cross-validation.
# 5. Calculate the validation score.
# 6. Repeat for the next combination.
# 7. Compare all scores.
# 8. Select the best combination.


grid_result = grid.fit(
    X_train,
    y_train
)


# ============================================================
# STEP 11: DISPLAY THE BEST RESULT
# ============================================================

# GridSearchCV stores the best average cross-validation score
# inside:
#
# grid_result.best_score_
#
#
# Example:
#
# 0.9475
#
# means the best hyperparameter combination achieved an average
# CV accuracy of approximately 94.75%.


# ------------------------------------------------------------
# BEST PARAMETERS
# ------------------------------------------------------------

# grid_result.best_params_
#
# Returns the hyperparameter combination that performed best.
#
# Example:
#
# {
#     'epochs': 100,
#     'layers': 2,
#     'neurons': 64
# }
#
# This means the best model configuration was:
#
# 64 neurons
# 2 hidden layers
# 100 epochs

print(
    "Best: %f using %s"
    % (
        grid_result.best_score_,
        grid_result.best_params_
    )
)


# ============================================================
#                    GRID SEARCH LOGIC
# ============================================================

# Our parameter grid contains:
#
# neurons -> 4 choices
# layers  -> 2 choices
# epochs  -> 2 choices
#
#
# Total combinations:
#
# 4 × 2 × 2 = 16 combinations
#
#
# Because cv=3:
#
# 16 combinations × 3 folds
# = 48 model trainings
#
#
# So GridSearchCV essentially does:
#
#
# neurons=16, layers=1, epochs=50
# neurons=16, layers=1, epochs=100
# neurons=16, layers=2, epochs=50
# neurons=16, layers=2, epochs=100
#
# neurons=32, layers=1, epochs=50
# neurons=32, layers=1, epochs=100
# neurons=32, layers=2, epochs=50
# neurons=32, layers=2, epochs=100
#
# neurons=64, layers=1, epochs=50
# neurons=64, layers=1, epochs=100
# neurons=64, layers=2, epochs=50
# neurons=64, layers=2, epochs=100
#
# neurons=128, layers=1, epochs=50
# neurons=128, layers=1, epochs=100
# neurons=128, layers=2, epochs=50
# neurons=128, layers=2, epochs=100
#
#
# Each combination is evaluated using 3-fold CV.
#
# Finally, GridSearchCV chooses the combination having the
# highest average validation score.